# Cas12a manuscript / sgRNA Modeler: 100 fixed-split trials

Run `Cas12a_create_and_export_fixed_split.ipynb` first.

This notebook loads the exact saved train, validation, and unseen tables.
It does not recreate the split.

The model is a modern scikit-learn implementation of the engineered-sequence
workflow used by `sgrna_modeler`:

- position-dependent mononucleotide indicators
- position-independent mononucleotide counts
- position-dependent dinucleotide indicators
- position-independent dinucleotide counts
- global GC count and fraction
- protospacer/PAM-region GC summaries when the context is long enough
- gradient-boosted regression trees

For each of 100 Optuna trials, it records:

- validation MSE
- unseen Pearson correlation
- unseen Spearman correlation

The best trial is selected by validation MSE only.


In [ ]:
from pathlib import Path
import hashlib
import json
import warnings

import joblib
import numpy as np
import optuna
import pandas as pd

from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# Original enPAM+GB feature implementation.
try:
    from sgrna_modeler import features as fe
    import sgrna_modeler.enzymes as en
except ImportError as exc:
    raise ImportError(
        "This notebook requires the original sgrna_modeler package/repository "
        "so that enPAM+GB uses its exact feature definitions. Install the "
        "repository in the active environment, for example with "
        "`pip install -e /path/to/sgrna_modeler`, then restart the kernel."
    ) from exc

OPTUNA_SEED = 42
MODEL_SEED = 42
N_TRIALS = 100

BASE_OUTPUT_DIR = Path("results/cas12a_kim2018_fixed_split")
SPLIT_DIR = BASE_OUTPUT_DIR / "saved_splits"
RESULTS_DIR = BASE_OUTPUT_DIR / "trial_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = SPLIT_DIR / "train_split.csv"
VALIDATION_FILE = SPLIT_DIR / "validation_split.csv"
UNSEEN_FILE = SPLIT_DIR / "unseen_split.csv"
MANIFEST_FILE = BASE_OUTPUT_DIR / "split_manifest.json"

for path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    UNSEEN_FILE,
    MANIFEST_FILE,
]:
    assert path.exists(), (
        f"Missing {path.resolve()}. "
        "Run the split notebook first."
    )

print("Results:", RESULTS_DIR.resolve())


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

with open(MANIFEST_FILE, "r") as handle:
    manifest = json.load(handle)

assert sha256(TRAIN_FILE) == manifest["train_sha256"]
assert sha256(VALIDATION_FILE) == manifest["validation_sha256"]
assert sha256(UNSEEN_FILE) == manifest["unseen_sha256"]

train_data = pd.read_csv(TRAIN_FILE)
validation_data = pd.read_csv(VALIDATION_FILE)
unseen_data = pd.read_csv(UNSEEN_FILE)

required_columns = {
    "target_context_sequence",
    "activity",
}

for name, frame in [
    ("train", train_data),
    ("validation", validation_data),
    ("unseen", unseen_data),
]:
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {sorted(missing)}")

print(
    len(train_data),
    len(validation_data),
    len(unseen_data),
)


In [ ]:
# Use the exact feature groups and Cas12a guide boundaries from the
# original enPAM+GB implementation in sgrna_modeler.models.SklearnSgrnaModel.
ENPAM_GB_FEATURES = [
    "Pos. Ind. 1mer",
    "Pos. Ind. 2mer",
    "Pos. Dep. 1mer",
    "Pos. Dep. 2mer",
    "GC content",
    "Tm",
]

CAS12A_GUIDE_START = en.cas12a["guide_start"]
CAS12A_GUIDE_LENGTH = en.cas12a["guide_length"]

def validate_sequences(sequence_series, split_name):
    """Normalize and validate sequences before original feature generation."""
    sequences = (
        sequence_series.astype(str)
        .str.strip()
        .str.upper()
    )

    invalid_mask = ~sequences.str.fullmatch(r"[ACGT]+")
    if invalid_mask.any():
        examples = sequences[invalid_mask].head(5).tolist()
        raise ValueError(
            f"{split_name} contains non-ACGT sequences. "
            f"Examples: {examples}"
        )

    lengths = sequences.str.len()
    if lengths.nunique() != 1:
        counts = lengths.value_counts().sort_index().to_dict()
        raise ValueError(
            f"{split_name} contains inconsistent sequence lengths: {counts}"
        )

    return sequences

def featurize_with_original_enpam_gb(sequence_series, split_name):
    """
    Generate the original enPAM+GB feature space.

    This delegates all positional/position-independent k-mer, GC-content,
    and melting-temperature calculations to sgrna_modeler.features rather
    than reimplementing them manually.
    """
    sequences = validate_sequences(sequence_series, split_name)

    feature_frame = fe.featurize_guides(
        sequences,
        features=ENPAM_GB_FEATURES,
        guide_start=CAS12A_GUIDE_START,
        guide_length=CAS12A_GUIDE_LENGTH,
    )

    # The original function normally returns a pandas DataFrame. Converting
    # defensively keeps the downstream sklearn/Optuna code unchanged.
    if not isinstance(feature_frame, pd.DataFrame):
        feature_frame = pd.DataFrame(feature_frame)

    feature_frame = feature_frame.reset_index(drop=True)

    if feature_frame.isnull().any().any():
        missing_columns = feature_frame.columns[
            feature_frame.isnull().any()
        ].tolist()
        raise ValueError(
            "Original enPAM+GB feature generation produced missing values "
            f"in columns: {missing_columns[:10]}"
        )

    return feature_frame

X_train = featurize_with_original_enpam_gb(
    train_data["target_context_sequence"],
    "train",
)
X_validation = featurize_with_original_enpam_gb(
    validation_data["target_context_sequence"],
    "validation",
)
X_unseen = featurize_with_original_enpam_gb(
    unseen_data["target_context_sequence"],
    "unseen",
)

# Enforce exactly the same feature names and ordering across all splits.
missing_validation = X_train.columns.difference(X_validation.columns)
extra_validation = X_validation.columns.difference(X_train.columns)
missing_unseen = X_train.columns.difference(X_unseen.columns)
extra_unseen = X_unseen.columns.difference(X_train.columns)

if (
    len(missing_validation)
    or len(extra_validation)
    or len(missing_unseen)
    or len(extra_unseen)
):
    raise ValueError(
        "Feature columns differ across saved splits. "
        f"Validation missing={list(missing_validation)}, "
        f"validation extra={list(extra_validation)}, "
        f"unseen missing={list(missing_unseen)}, "
        f"unseen extra={list(extra_unseen)}"
    )

X_validation = X_validation.loc[:, X_train.columns]
X_unseen = X_unseen.loc[:, X_train.columns]

y_train = train_data["activity"].to_numpy(dtype=float)
y_validation = validation_data["activity"].to_numpy(dtype=float)
y_unseen = unseen_data["activity"].to_numpy(dtype=float)

print("Original enPAM+GB feature groups:", ENPAM_GB_FEATURES)
print(
    "Cas12a guide_start / guide_length:",
    CAS12A_GUIDE_START,
    CAS12A_GUIDE_LENGTH,
)
print("Feature count:", X_train.shape[1])
print(X_train.shape, X_validation.shape, X_unseen.shape)


In [ ]:
def safe_pearson(y_true, y_pred):
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(pearsonr(y_true, y_pred)[0])

def safe_spearman(y_true, y_pred):
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(spearmanr(y_true, y_pred)[0])


In [ ]:
trial_records = []
trial_models = {}

def objective(trial):
    model = GradientBoostingRegressor(
        loss="squared_error",
        random_state=MODEL_SEED,
        n_estimators=trial.suggest_int(
            "n_estimators", 50, 500
        ),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.20, log=True
        ),
        max_depth=trial.suggest_int(
            "max_depth", 2, 6
        ),
        min_samples_split=trial.suggest_int(
            "min_samples_split", 2, 30
        ),
        min_samples_leaf=trial.suggest_int(
            "min_samples_leaf", 1, 20
        ),
        max_features=trial.suggest_categorical(
            "max_features", [None, "sqrt", "log2"]
        ),
        subsample=trial.suggest_float(
            "subsample", 0.60, 1.00
        ),
    )

    model.fit(X_train, y_train)

    validation_predictions = model.predict(X_validation)
    unseen_predictions = model.predict(X_unseen)

    validation_mse = float(
        mean_squared_error(
            y_validation,
            validation_predictions,
        )
    )

    unseen_pearson = safe_pearson(
        y_unseen,
        unseen_predictions,
    )

    unseen_spearman = safe_spearman(
        y_unseen,
        unseen_predictions,
    )

    trial_records.append({
        "trial": trial.number,
        "validation_mse": validation_mse,
        "unseen_pearson": unseen_pearson,
        "unseen_spearman": unseen_spearman,
    })

    trial_models[trial.number] = model

    print(
        f"Trial {trial.number:3d} | "
        f"Validation MSE: {validation_mse:.6f} | "
        f"Unseen Pearson: {unseen_pearson:.4f} | "
        f"Unseen Spearman: {unseen_spearman:.4f}"
    )

    return validation_mse

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=OPTUNA_SEED),
    study_name="Cas12a_Kim2018_fixed_split_100_trials",
)

study.optimize(objective, n_trials=N_TRIALS)


In [ ]:
results_df = (
    pd.DataFrame(trial_records)
    .sort_values("trial")
    .reset_index(drop=True)
)

results_df.to_csv(
    RESULTS_DIR / "all_100_trial_metrics.csv",
    index=False,
)
results_df["validation_mse"].to_csv(
    RESULTS_DIR / "Validation_loss.txt",
    index=False,
    header=False,
)
results_df["unseen_pearson"].to_csv(
    RESULTS_DIR / "Unseen_Pearson.txt",
    index=False,
    header=False,
)
results_df["unseen_spearman"].to_csv(
    RESULTS_DIR / "Unseen_Spearman.txt",
    index=False,
    header=False,
)

best_trial = study.best_trial.number
best_row = results_df.loc[
    results_df["trial"] == best_trial
].iloc[0]

joblib.dump(
    {
        "model": trial_models[best_trial],
        "feature_columns": X_train.columns.tolist(),
        "best_trial": int(best_trial),
        "best_params": study.best_trial.params,
        "split_manifest": manifest,
        "validation_mse": float(best_row["validation_mse"]),
        "unseen_pearson": float(best_row["unseen_pearson"]),
        "unseen_spearman": float(best_row["unseen_spearman"]),
    },
    RESULTS_DIR / "best_Cas12a_model.joblib",
)

with open(RESULTS_DIR / "best_trial_summary.json", "w") as handle:
    json.dump(
        {
            "best_trial": int(best_trial),
            "best_params": study.best_trial.params,
            "validation_mse": float(best_row["validation_mse"]),
            "unseen_pearson": float(best_row["unseen_pearson"]),
            "unseen_spearman": float(best_row["unseen_spearman"]),
        },
        handle,
        indent=2,
    )

display(results_df.head())
print("Best trial:", best_trial)
print(best_row)
print("Saved to:", RESULTS_DIR.resolve())
